# LLaVA-1.5-7B · MVTec AD Fine-Tuning

**Single notebook — 8 sections.** Run top-to-bottom for a fresh run.  
If Colab disconnects, re-run **sections 1–3** then jump straight to **section 6** — `Trainer` resumes from the last saved checkpoint automatically.

| Section | What it does |
|---------|-------------|
| 1 | Mount Google Drive (checkpoint persistence) |
| 2 | Install dependencies |
| 3 | Clone repo & configure |
| 4 | Download MVTec AD via Kaggle API |
| 5 | Prepare training JSON |
| 6 | Train (auto-resumes from last checkpoint) |
| 7 | Evaluate — accuracy / F1 / confusion matrix |
| 8 | Quick inference demo |

## Section 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Checkpoints and data will be stored here so they survive disconnects.
DRIVE_ROOT = '/content/drive/MyDrive/vlm-defect-detection'

import os
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
print('Drive mounted. Project root:', DRIVE_ROOT)

## Section 2 — Install Dependencies

In [ ]:
# Clone the repo (skip if already present)
import os
if not os.path.exists('/content/vlm-defect-detection'):
    !git clone https://github.com/Pranavk098/vlm-defect-detection.git /content/vlm-defect-detection

%cd /content/vlm-defect-detection

# Install the training extras — no LLaVA repo clone needed
!pip install -q -e '.[train]'

# Optional: Flash Attention 2 saves ~1 GB VRAM on A100
# !pip install -q -e '.[train,flash]'

print('Installation complete.')

## Section 3 — Configure W&B and Symlink Drive Directories

In [ ]:
import os

# ── W&B (optional but recommended) ─────────────────────────────────────────
# Paste your W&B API key here or leave blank to skip.
WANDB_API_KEY = ''  # @param {type: 'string'}

if WANDB_API_KEY:
    os.environ['WANDB_API_KEY'] = WANDB_API_KEY
    !wandb login --relogin
    REPORT_TO = 'wandb'
else:
    REPORT_TO = 'none'
    print('W&B skipped. Set WANDB_API_KEY above to enable loss monitoring.')

# ── Symlink Drive → /content for fast local I/O ─────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/vlm-defect-detection'

for name in ('checkpoints', 'data'):
    local = f'/content/vlm-defect-detection/{name}'
    remote = f'{DRIVE_ROOT}/{name}'
    os.makedirs(remote, exist_ok=True)
    if not os.path.islink(local):
        if os.path.exists(local):
            import shutil; shutil.rmtree(local)
        os.symlink(remote, local)
        print(f'Symlinked {local} → {remote}')

print('Configuration complete.')

## Section 4 — Download MVTec AD via Kaggle API

In [ ]:
# Upload your kaggle.json here, or paste credentials below.
# Get it from: https://www.kaggle.com/settings → API → Create New Token

KAGGLE_USERNAME = ''  # @param {type: 'string'}
KAGGLE_KEY      = ''  # @param {type: 'string'}

import os, json

if KAGGLE_USERNAME and KAGGLE_KEY:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    creds = {'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
        json.dump(creds, f)
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    !pip install -q kaggle
    !kaggle datasets download -d ipythonx/mvtec-ad --unzip -p /content/vlm-defect-detection/
    # Rename to the expected directory name if needed
    import glob
    candidates = glob.glob('/content/vlm-defect-detection/mvtec*')
    print('Dataset directories found:', candidates)
else:
    print('Provide Kaggle credentials above, or manually upload and extract MVTec AD')
    print('Expected path: /content/vlm-defect-detection/mvtec_anomaly_detection/')

## Section 5 — Prepare Training JSON

In [ ]:
%cd /content/vlm-defect-detection

import os
if os.path.exists("data/mvtec_train.json") and os.path.exists("data/mvtec_test.json"):
    print("Both JSON splits already exist — skipping prepare step.")
else:
    !vlm-prepare --dataset-root mvtec_anomaly_detection --output-train data/mvtec_train.json --output-test data/mvtec_test.json

import json
with open("data/mvtec_train.json") as f:
    n_train = len(json.load(f))
with open("data/mvtec_test.json") as f:
    n_test = len(json.load(f))
print(f"{n_train} train samples, {n_test} test samples.")


## Section 6 — Train

> **Reconnect after disconnect?** Re-run sections 1–3, then run this cell.  
> `Trainer` detects the latest checkpoint in `checkpoints/llava-mvtec-lora/`  
> and resumes automatically — no steps are repeated.

In [ ]:
%cd /content/vlm-defect-detection

# REPORT_TO was set in section 3 ("wandb" or "none")
override = f"training.report_to={REPORT_TO}"

!vlm-train configs/local_8gb.yaml {override}


## Section 7 — Evaluate

In [ ]:
%cd /content/vlm-defect-detection

CHECKPOINT = 'checkpoints/llava-mvtec-lora'  # @param {type: 'string'}

!python scripts/evaluate.py {CHECKPOINT} configs/local_8gb.yaml

import json, pathlib
results_path = pathlib.Path(CHECKPOINT) / 'eval_results.json'
if results_path.exists():
    results = json.loads(results_path.read_text())
    print('\n── Summary ──')
    for k, v in results.items():
        print(f'  {k}: {v}')

## Section 8 — Quick Inference Demo

In [ ]:
%cd /content/vlm-defect-detection

import torch
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration
from peft import PeftModel

CHECKPOINT = 'checkpoints/llava-mvtec-lora'  # @param {type: 'string'}
IMAGE_PATH = ''  # @param {type: 'string'}  ← path to a test image

base_id = 'llava-hf/llava-1.5-7b-hf'
processor = AutoProcessor.from_pretrained(base_id)
model = LlavaForConditionalGeneration.from_pretrained(
    base_id, torch_dtype=torch.bfloat16, device_map='auto'
)
model = PeftModel.from_pretrained(model, CHECKPOINT)
model.eval()

if IMAGE_PATH:
    image = Image.open(IMAGE_PATH).convert('RGB')
    prompt = 'USER: <image>\nIs there any anomaly in this image? Describe it. ASSISTANT:'
    inputs = processor(text=prompt, images=image, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    answer = processor.tokenizer.decode(
        out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True
    )
    print('Model:', answer)
else:
    print('Set IMAGE_PATH above to run inference on a specific image.')